# VIX Market Crash — V5 : XGBoost macro + futures VX1–VX4\n
**Objectif** : prédire une baisse du S&P 500 de 2 % ou plus entre t et t+5, à partir de la courbe du VIX et de cinq marchés supplémentaires : **EUR/USD, or, pétrole WTI, Treasury US 3 ans, rendement corporate high yield US**.

**Sources** : VIX et SPX depuis Google Sheets (identiques à V3) ; DEXUSEU, DCOILWTICO, DGS3 et BAMLH0A0HYM2 depuis FRED ; or via l'ETF GLD (Stooq, proxy). Les données externes sont **décalées d'une séance** après alignement afin d'éviter de présumer qu'elles étaient publiées au moment de la clôture SPX t. Les trous courts (jours fériés) sont comblés, pas les longues périodes sans données.

**Comparaison** : XGBoost VIX seul (baseline V3), XGBoost VIX + macro, XGBoost VIX + macro + probabilité RF OOF. Les modèles utilisent exactement les mêmes dates train/test. Cible prospective avec purge de 5 séances à la frontière train/test. Attention aux fenêtres de rendement qui se chevauchent et à la disponibilité effective des données FRED.

## 0. Imports et paramètres

In [ ]:
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import google.auth
from google.auth.transport.requests import AuthorizedSession
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

STRATEGY_SHEET_ID = '1XKeBA6hcnPLcdPixX5bWBtbDTBWP-MtK7Av8ead0NN0'
CRASH_DAYS = 5
CRASH_FLAG = -0.02
TEST_FRACTION = 0.20
RANDOM_STATE = 42
VIX_FEATURES = ['Slope_1M_3M', 'Slope_3M_6M', 'Slope_6M_1Y',
                'VIX1M_Change', 'VIX3M_Change', 'VIX6M_Change', 'VIX1Y_Change']
MACRO_FEATURES = [
    'VIX1M_Level', 'SPX_Return_Past5D',
    'EURUSD_Level', 'EURUSD_Return_5D',
    'Gold_Return_5D', 'WTI_Return_5D',
    'UST3Y_Yield', 'UST3Y_Change_5D',
    'USHY_Yield', 'USHY_Change_5D', 'HY_minus_UST3Y'
]
ALL_FEATURES = VIX_FEATURES + MACRO_FEATURES

## 1. Google Sheets + FRED : chargement et alignement

In [ ]:
def reload_strategy_excel():
    credentials, _ = google.auth.default(
        scopes=['https://www.googleapis.com/auth/drive.readonly'])
    session = AuthorizedSession(credentials)
    response = session.get(
        f'https://www.googleapis.com/drive/v3/files/{STRATEGY_SHEET_ID}/export',
        params={'mimeType': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'},
        timeout=120)
    response.raise_for_status()
    return BytesIO(response.content)

def align_df_dates(raw, names):
    if raw.shape[1] != 2 * len(names):
        raise ValueError(f'Attendu {2 * len(names)} colonnes Date/Valeur, reçu {raw.shape[1]}')
    aligned = []
    for i, name in enumerate(names):
        part = raw.iloc[:, [2*i, 2*i+1]].copy()
        part.columns = ['Date', name]
        part['Date'] = pd.to_datetime(part['Date'], errors='coerce').dt.normalize()
        part[name] = pd.to_numeric(part[name], errors='coerce')
        part = part.dropna(subset=['Date', name]).drop_duplicates('Date', keep='last')
        aligned.append(part.set_index('Date'))
    return pd.concat(aligned, axis=1, join='outer').sort_index()

strategy_excel = reload_strategy_excel()
data_raw = pd.read_excel(strategy_excel, sheet_name='VIX HISTORIQUE',
                         skiprows=3, usecols=[0,1,2,3,5,6,7,8,9,10])
data = align_df_dates(data_raw, ['SPX','VIX1M','VIX3M','VIX6M','VIX1Y'])
data = data.loc[data['SPX'].notna()].copy()
print('Couverture VIX par série (union des dates, sans intersection prématurée):')
print(data[['SPX','VIX1M','VIX3M','VIX6M','VIX1Y']].apply(lambda x: (x.first_valid_index(), x.last_valid_index())).to_string())
assert len(data) > 100 and data['SPX'].notna().sum() > 100
assert not data.index.has_duplicates and data.index.is_monotonic_increasing

def fred_series(series_id):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd=1990-01-01&coed=2099-12-31'
    frame = pd.read_csv(url, na_values=['.'])
    frame.columns = ['Date', series_id]
    frame['Date'] = pd.to_datetime(frame['Date'])
    frame[series_id] = pd.to_numeric(frame[series_id], errors='coerce')
    series = frame.set_index('Date')[series_id].sort_index()
    last = series.last_valid_index()
    if last is None or last < pd.Timestamp.today().normalize() - pd.Timedelta(days=120):
        raise ValueError(
            f"FRED {series_id}: dernière valeur {last}; série périmée. "
            "Vérifier l'URL et la disponibilité chez FRED avant entraînement.")
    return series

def gold_prices():
    # Source fiable pour ce projet : MACRO HISTORIQUE, C=Date GLD, D=GLD USD.
    # Les dates GoogleFinance incluent une heure, normalisee a la journee.
    raw = pd.read_excel(strategy_excel, sheet_name='MACRO HISTORIQUE',
                        skiprows=5, usecols='C:D', header=None,
                        names=['Date', 'Gold'])
    raw['Date'] = pd.to_datetime(raw['Date'], errors='coerce').dt.normalize()
    raw['Gold'] = pd.to_numeric(raw['Gold'], errors='coerce')
    raw = raw.dropna(subset=['Date','Gold']).drop_duplicates('Date', keep='last')
    gold = raw.set_index('Date')['Gold'].sort_index()
    if gold.notna().sum() < 500:
        raise ValueError('Historique GLD de MACRO HISTORIQUE insuffisant')
    print('Or : GLD, colonne D de MACRO HISTORIQUE (dates colonne C).')
    return gold

hy_sheet = pd.read_excel(strategy_excel, sheet_name='FRED_US_HY_Spread', usecols='A:B')
hy_sheet.columns = ['Date', 'USHY']
hy_sheet['Date'] = pd.to_datetime(hy_sheet['Date'], errors='coerce').dt.normalize()
hy_sheet['USHY'] = pd.to_numeric(hy_sheet['USHY'], errors='coerce')
hy_sheet = (hy_sheet.dropna(subset=['Date', 'USHY'])
            .drop_duplicates('Date', keep='last').set_index('Date')['USHY'].sort_index())
# Conserver l'historique du Sheet (depuis 1996), completer les dates recentes
# par FRED ; les valeurs FRED ont priorite lorsqu'elles existent.
hy_fred = fred_series('BAMLH0A0HYM2')
hy_full = hy_fred.combine_first(hy_sheet).sort_index()
print('USHY : Sheet', hy_sheet.first_valid_index(), '→', hy_sheet.last_valid_index(),
      '| fusion FRED →', hy_full.last_valid_index())
external_raw = pd.DataFrame({
    'EURUSD': fred_series('DEXUSEU'),
    'Gold': gold_prices(),
    'WTI': fred_series('DCOILWTICO'),
    'UST3Y': fred_series('DGS3'),
    'USHY': hy_full,
})
# Vérifier la couverture historique : la série or peut s'arrêter avant les autres.
print('Dernière observation disponible par série FRED :')
display(external_raw.apply(lambda col: col.last_valid_index()).to_frame('Dernière date'))
# Alignement sur les séances SPX, remplissage limité aux jours fériés.
# Shift(1) = valeurs externes observées au plus tard à la séance SPX précédente.
external = external_raw.reindex(data.index, method='ffill', limit=3).shift(1)
data = data.join(external)
print('Observations SPX/VIX :', len(data), 'du', data.index.min().date(),
      'au', data.index.max().date())
print('Disponibilité macro :')
display(data[['EURUSD','Gold','WTI','UST3Y','USHY']].notna().sum().to_frame('N séances'))
display(data.tail())
if data['Gold'].notna().sum() < 500:
    print('Or indisponible : la V5 continuera sans Gold_Return_5D.')
# Bloquer tout entraînement avec des séries macro tronquées.
for key in ['EURUSD','WTI','UST3Y','USHY']:
    last = data[key].last_valid_index()
    if last is None or last < data.index.max() - pd.Timedelta(days=120):
        raise ValueError(f'Données macro périmées : {key}, dernière date={last}')

## 2. Exploration des données

In [ ]:
display(data.describe().T)
print('Valeurs manquantes :\n', data.isna().sum())
print('Corrélations :')
display(data.corr().round(3))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
data['SPX'].plot(ax=axes[0], title='S&P 500')
(data['VIX1M'] - data['VIX3M']).plot(ax=axes[1], title='Pente VIX 1M - 3M')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

## 3. Variables VIX + cross-asset, cibles et découpage chronologique

In [ ]:
vix_data = data.copy()
vix_data['Slope_1M_3M'] = data['VIX1M'] - data['VIX3M']
vix_data['Slope_3M_6M'] = data['VIX3M'] - data['VIX6M']
vix_data['Slope_6M_1Y'] = data['VIX6M'] - data['VIX1Y']
for name in ['VIX1M','VIX3M','VIX6M','VIX1Y']:
    vix_data[name + '_Change'] = data[name].diff(CRASH_DAYS)

vix_data['SPX_Return_5D'] = data['SPX'].pct_change(CRASH_DAYS)
vix_data['VIX1M_Level'] = data['VIX1M']
vix_data['SPX_Return_Past5D'] = vix_data['SPX_Return_5D']
vix_data['EURUSD_Level'] = data['EURUSD']
vix_data['EURUSD_Return_5D'] = data['EURUSD'].pct_change(CRASH_DAYS, fill_method=None)
vix_data['Gold_Return_5D'] = data['Gold'].pct_change(CRASH_DAYS, fill_method=None)
vix_data['WTI_Return_5D'] = data['WTI'].pct_change(CRASH_DAYS, fill_method=None)
vix_data['UST3Y_Yield'] = data['UST3Y']
vix_data['UST3Y_Change_5D'] = data['UST3Y'].diff(CRASH_DAYS)
vix_data['USHY_Yield'] = data['USHY']
vix_data['USHY_Change_5D'] = data['USHY'].diff(CRASH_DAYS)
vix_data['HY_minus_UST3Y'] = data['USHY'] - data['UST3Y']

# Exclure uniquement la variable or si le fournisseur alternatif est indisponible.
if vix_data['Gold_Return_5D'].notna().sum() < 500:
    MACRO_FEATURES.remove('Gold_Return_5D')
    ALL_FEATURES = VIX_FEATURES + MACRO_FEATURES
    print('ATTENTION : modèle sans or (données GLD indisponibles).')

# Dates communes à tous les modèles ; n'éliminer que les features manquantes.
vix_data = vix_data.dropna(subset=ALL_FEATURES).copy()
print('Échantillon complet VIX 1M/3M/6M/1Y :', len(vix_data), 'séances; le modèle historique 2007+ exige un jeu de features réduit.')
if len(vix_data) < 500:
    raise ValueError(f'Seulement {len(vix_data)} séances communes : vérifier Yahoo/FRED.')
split = int(len(vix_data) * (1 - TEST_FRACTION))
train_dates = vix_data.index[:split]
test_dates = vix_data.index[split:]
thresholds = vix_data.loc[train_dates, VIX_FEATURES].abs().quantile(.75)
vix_data['VIX_Significant_Change'] = vix_data[VIX_FEATURES].abs().gt(thresholds).any(axis=1).astype(int)
vix_data['VIX_Target_T1'] = vix_data['VIX_Significant_Change'].shift(-1)

# IMPORTANT : cible future calculée sur le calendrier SPX original,
# AVANT le filtrage des dates macro (évite de déformer l'horizon de 5 séances).
spx_fwd = data['SPX'].shift(-CRASH_DAYS) / data['SPX'] - 1
vix_data['SPX_Return_Fwd5D'] = spx_fwd.reindex(vix_data.index)
vix_data['Crash_Target_Fwd5D'] = (
    vix_data['SPX_Return_Fwd5D'].le(CRASH_FLAG).astype(float)
    .where(vix_data['SPX_Return_Fwd5D'].notna()))
print('Train :', train_dates[0].date(), '→', train_dates[-1].date())
print('Test :', test_dates[0].date(), '→', test_dates[-1].date())
print('Variables XGB :', len(ALL_FEATURES))
display(thresholds.rename('Seuil 75 % (train seulement)').to_frame())

## 4. Random Forest : test chronologique et prédictions OOF

In [ ]:
# Le dernier point du train est exclu : sa cible T+1 est située dans le test.
rf_train = vix_data.iloc[:split-1].dropna(subset=['VIX_Target_T1'])
rf_test = vix_data.iloc[split:].dropna(subset=['VIX_Target_T1'])
stage1_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
stage1_model.fit(rf_train[VIX_FEATURES], rf_train['VIX_Target_T1'].astype(int))
rf_test_proba = stage1_model.predict_proba(rf_test[VIX_FEATURES])[:,1]
print('Random Forest AUC test :', round(roc_auc_score(rf_test['VIX_Target_T1'], rf_test_proba), 3))
print(classification_report(rf_test['VIX_Target_T1'].astype(int), rf_test_proba >= .5, zero_division=0))

# OOF avec 5 plis chronologiques, et gap=1 pour que les labels T+1 du train
# ne contiennent pas d'observations de la fenêtre de validation.
oof = pd.Series(np.nan, index=vix_data.index, dtype=float)
tscv = TimeSeriesSplit(n_splits=5, gap=1)
for idx_train, idx_val in tscv.split(vix_data.iloc[:split]):
    fit_rows = vix_data.iloc[idx_train].dropna(subset=['VIX_Target_T1'])
    val_rows = vix_data.iloc[idx_val]
    if fit_rows['VIX_Target_T1'].nunique() < 2:
        continue
    fold_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    fold_rf.fit(fit_rows[VIX_FEATURES], fit_rows['VIX_Target_T1'].astype(int))
    oof.loc[val_rows.index] = fold_rf.predict_proba(val_rows[VIX_FEATURES])[:,1]

vix_data['RF_Probability_T1'] = oof
vix_data.loc[rf_test.index, 'RF_Probability_T1'] = rf_test_proba
# La dernière observation n'a pas de label RF test, mais sa probabilité peut être calculée.
if vix_data.index[-1] in test_dates:
    vix_data.loc[vix_data.index[-1], 'RF_Probability_T1'] = (
        stage1_model.predict_proba(vix_data.iloc[[-1]][VIX_FEATURES])[0,1])
print('Prédictions OOF pour XGB :', oof.notna().sum())

## 5. XGBoost : VIX seul / VIX + macro / VIX + macro + RF

In [ ]:
# Même échantillon pour les 3 modèles.
# Purge temporelle sur le calendrier SPX, pas simplement 5 lignes du jeu filtré.
test_start = test_dates[0]
train_end_safe = data.index[data.index.get_loc(test_start) - CRASH_DAYS]
features_with_rf = ALL_FEATURES + ['RF_Probability_T1']
stage2_train = vix_data.loc[vix_data.index < train_end_safe].dropna(
    subset=features_with_rf + ['Crash_Target_Fwd5D'])
stage2_test = vix_data.loc[vix_data.index >= test_start].dropna(
    subset=features_with_rf + ['Crash_Target_Fwd5D'])
y_train = stage2_train['Crash_Target_Fwd5D'].astype(int)
y_test = stage2_test['Crash_Target_Fwd5D'].astype(int)
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Cible crash : il faut les deux classes en train et test.')
weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

def make_xgb():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=.05,
                         random_state=RANDOM_STATE, scale_pos_weight=weight,
                         eval_metric='logloss', n_jobs=-1)

feature_sets = {
    'VIX seul (baseline)': VIX_FEATURES,
    'VIX + macro': ALL_FEATURES,
    'VIX + macro + RF': features_with_rf,
}
fitted_models = {}
test_probabilities = {}
for label, cols in feature_sets.items():
    model = make_xgb()
    model.fit(stage2_train[cols], y_train)
    fitted_models[label] = model
    test_probabilities[label] = model.predict_proba(stage2_test[cols])[:, 1]
    print(f'{label} : AUC test = {roc_auc_score(y_test, test_probabilities[label]):.3f}')

auc_base = roc_auc_score(y_test, test_probabilities['VIX seul (baseline)'])
for label in ['VIX + macro', 'VIX + macro + RF']:
    print(f'Gain {label} vs baseline : '
          f'{roc_auc_score(y_test, test_probabilities[label]) - auc_base:+.3f}')
print('\nRapport VIX + macro :')
print(classification_report(y_test, test_probabilities['VIX + macro'] >= .5,
                            zero_division=0))
results_v4 = pd.DataFrame({'Actual_Crash_Fwd5D': y_test,
                           'Realized_SPX_Fwd5D': stage2_test['SPX_Return_Fwd5D']},
                          index=stage2_test.index)
for label, proba in test_probabilities.items():
    results_v4['Probability_' + label] = proba
display(results_v4.tail())

print('\nImportance des variables (gain XGBoost, indicatif) :')
importance = pd.Series(fitted_models['VIX + macro'].feature_importances_,
                       index=ALL_FEATURES).sort_values(ascending=False)
display(importance.to_frame('importance').head(15))

## 6. Backtest indicatif VIX + macro (blocs non chevauchants)
Signal après la clôture t ; entrée hypothétique à la clôture t+1, sortie t+6. Les rendements par blocs de 5 séances ne se chevauchent pas. Comparaison simplifiée, sans frais, slippage ni rémunération du cash.

In [ ]:
bt = results_v4.copy()
bt['Tradable_5D_Return'] = (
    data['SPX'].shift(-(CRASH_DAYS+1)) / data['SPX'].shift(-1) - 1
).reindex(bt.index)
bt = bt.dropna(subset=['Tradable_5D_Return']).iloc[::CRASH_DAYS].copy()
bt['Invested'] = (bt['Probability_VIX + macro'] < .5).astype(int)
bt['Strategy_Return'] = bt['Invested'] * bt['Tradable_5D_Return']
bt['SPX_BuyHold'] = (1 + bt['Tradable_5D_Return']).cumprod()
bt['Strategy'] = (1 + bt['Strategy_Return']).cumprod()
bt[['SPX_BuyHold','Strategy']].plot(figsize=(12,5),
    title='V4 — Backtest indicatif VIX + macro, blocs de 5 séances')
plt.ylabel('Capital normalisé')
plt.grid(True)
plt.show()
print('Blocs :', len(bt))
print('Part investie :', f"{bt['Invested'].mean():.1%}")
print('Performance stratégie :', f"{bt['Strategy'].iloc[-1]-1:.1%}")
print('Performance buy & hold :', f"{bt['SPX_BuyHold'].iloc[-1]-1:.1%}")

## 7. Modèle historique V5 : XGBoost macro + VX1/VX2 depuis 2004\n\nLes premiers futures VIX sont disponibles à partir de mars 2004, mais les quatre échéances ne sont pas systématiquement présentes. Le modèle historique utilise VX1, VX2, leur pente et leur contango. L'échantillon effectif commence à la première date où toutes les variables macro requises sont présentes (notamment GLD à partir de novembre 2004). Les prix futures sont décalés d'une séance et la cible est purgée de cinq séances.\n

In [ ]:
from pathlib import Path
from sklearn.metrics import average_precision_score, brier_score_loss
VX_PATH = Path("vix_futures_cboe/VIX_FUTURES_HISTORIQUE.csv")
ANOMALY_PATH = Path("vix_futures_cboe/VIX_FUTURES_ANOMALIES.csv")
if not VX_PATH.exists():
    raise FileNotFoundError("Lancer python3 build_vix_futures.py dans le dépôt")
vx = pd.read_csv(VX_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
for i in range(1, 5):
    vx[f"VX{i}_Expiry"] = pd.to_datetime(vx[f"VX{i}_Expiry"])
    # Différence quotidienne valide uniquement à expiration identique.
    same_contract = vx[f"VX{i}_Expiry"].eq(vx[f"VX{i}_Expiry"].shift(1))
    vx[f"VX{i}_Change_SameContract"] = vx[f"VX{i}"].diff().where(same_contract)
    vx[f"VX{i}_Pct_SameContract"] = vx[f"VX{i}"].pct_change(fill_method=None).where(same_contract)
vx["VX_Slope_1_2"] = vx["VX2"] - vx["VX1"]
vx["VX_Slope_2_3"] = vx["VX3"] - vx["VX2"]
vx["VX_Slope_3_4"] = vx["VX4"] - vx["VX3"]
vx["VX_Contango_1_2"] = vx["VX2"] / vx["VX1"] - 1
if ANOMALY_PATH.exists():
    anomalies = pd.read_csv(ANOMALY_PATH)
    print("Alertes de qualité:", len(anomalies))
    display(anomalies.groupby(["Series", "Anomaly"]).size().rename("Count").to_frame())
else:
    print("Audit absent : exécuter python3 audit_vix_futures.py")
# Disponibilité à t: décaler les settlements d'une séance par prudence.
VX_MODEL_FEATURES = ["VX1", "VX2", "VX3", "VX4", "VX_Slope_1_2",
                     "VX_Slope_2_3", "VX_Slope_3_4", "VX_Contango_1_2"]
# Exclusion provisoire des cotations manifestement suspectes, sans toucher aux CSV bruts.
# Les observations 2006 ne sont pas utilisées dans le modèle futures (début 2007).
suspect_dates = pd.to_datetime([])  # échelle corrigée dans build_vix_futures.py
vx.loc[vx.index.isin(suspect_dates), VX_MODEL_FEATURES] = np.nan
# Les variations de 2018/2020/2024 sont conservées : elles peuvent être réelles.
vx_features = vx[VX_MODEL_FEATURES].shift(1)
vx_features = vx_features.rename(columns=lambda c: "FUT_" + c)
# Ne pas forward-fill les trous de contrats.
vx_features = vx_features.reindex(data.index)
print("Couverture futures sur les dates SPX disponibles:")
display(vx_features.notna().groupby(vx_features.index.year).mean().round(3).tail(23))

# Comparaison historique : ne pas imposer VIX3M/6M/1Y disponibles seulement en 2018.
# Les features VX sont toujours décalees d'une seance pour eviter le lookahead.
HISTORICAL_FEATURES = [
    'VIX1M_Level', 'VIX1M_Change', 'SPX_Return_Past5D',
    'EURUSD_Level', 'EURUSD_Return_5D', 'Gold_Return_5D',
    'WTI_Return_5D', 'UST3Y_Yield', 'UST3Y_Change_5D',
    'USHY_Yield', 'USHY_Change_5D', 'HY_minus_UST3Y'
]
historical = data.copy()
historical['VIX1M_Level'] = data['VIX1M']
historical['VIX1M_Change'] = data['VIX1M'].diff(CRASH_DAYS)
historical['SPX_Return_Past5D'] = data['SPX'].pct_change(CRASH_DAYS)
historical['EURUSD_Level'] = data['EURUSD']
historical['EURUSD_Return_5D'] = data['EURUSD'].pct_change(CRASH_DAYS, fill_method=None)
historical['Gold_Return_5D'] = data['Gold'].pct_change(CRASH_DAYS, fill_method=None)
historical['WTI_Return_5D'] = data['WTI'].where(data['WTI'] > 0).pct_change(CRASH_DAYS, fill_method=None)
historical['UST3Y_Yield'] = data['UST3Y']
historical['UST3Y_Change_5D'] = data['UST3Y'].diff(CRASH_DAYS)
historical['USHY_Yield'] = data['USHY']
historical['USHY_Change_5D'] = data['USHY'].diff(CRASH_DAYS)
historical['HY_minus_UST3Y'] = data['USHY'] - data['UST3Y']
historical['SPX_Return_Fwd5D'] = data['SPX'].shift(-CRASH_DAYS) / data['SPX'] - 1
historical['Crash_Target_Fwd5D'] = historical['SPX_Return_Fwd5D'].le(CRASH_FLAG).astype(float).where(historical['SPX_Return_Fwd5D'].notna())
# USHY conservee : source historique FRED_US_HY_Spread depuis 1996.
print('Couverture USHY historique:', historical['USHY_Yield'].notna().sum(), 'séances')
vx_features = vx_features.reindex(data.index)
# VX3/VX4 peuvent manquer en 2004-2005 : conserver seulement les
# deux premiers contrats pour le modèle historique dès 2004.
EARLY_VX_FEATURES = ['FUT_VX1', 'FUT_VX2', 'FUT_VX_Slope_1_2', 'FUT_VX_Contango_1_2']
print('Première date VX1/VX2 utilisable:', vx_features[EARLY_VX_FEATURES].dropna().index.min())
futures_dataset = historical.join(vx_features, how='left')
futures_dataset = futures_dataset.loc['2004-03-26':].dropna(
    subset=HISTORICAL_FEATURES + EARLY_VX_FEATURES + ['Crash_Target_Fwd5D'])
print('Période effective:', futures_dataset.index.min(), '→', futures_dataset.index.max(), '| séances:', len(futures_dataset))
if len(futures_dataset) < 1000:
    raise ValueError(f'Échantillon futures insuffisant : {len(futures_dataset)} lignes')
boundary = int(len(futures_dataset) * (1 - TEST_FRACTION))
test_start = futures_dataset.index[boundary]
safe_train_end = data.index[data.index.get_loc(test_start) - CRASH_DAYS]
train = futures_dataset.loc[futures_dataset.index < safe_train_end]
test = futures_dataset.loc[futures_dataset.index >= test_start]
y_train = train['Crash_Target_Fwd5D'].astype(int)
y_test = test['Crash_Target_Fwd5D'].astype(int)
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Cible crash : classes insuffisantes en train ou test')
def fit_futures_model(features):
    model = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=.035,
                          subsample=.8, colsample_bytree=.85,
                          eval_metric='logloss', random_state=RANDOM_STATE)
    model.fit(train[features], y_train)
    probabilities = model.predict_proba(test[features])[:, 1]
    auc = roc_auc_score(y_test, probabilities)
    return model, auc, probabilities
base_model, base_auc, base_probs = fit_futures_model(HISTORICAL_FEATURES)
futures_model, futures_auc, futures_probs = fit_futures_model(
    HISTORICAL_FEATURES + EARLY_VX_FEATURES)
print('Comparaison historique VX1/VX2 depuis 2004 à dates identiques :')
print(f'Macro sans futures : AUC={base_auc:.4f}')
print(f'Macro + VX1–VX4 : AUC={futures_auc:.4f}')
print(f'Macro seule : AP={average_precision_score(y_test, base_probs):.4f}, Brier={brier_score_loss(y_test, base_probs):.4f}')
print(f'Macro + futures : AP={average_precision_score(y_test, futures_probs):.4f}, Brier={brier_score_loss(y_test, futures_probs):.4f}')
print('Train:', train.index.min().date(), '→', train.index.max().date())
print('Test :', test.index.min().date(), '→', test.index.max().date())
comparison_v5 = pd.DataFrame({'Date':test.index, 'Crash_Observed':y_test.values,
    'Prob_Macro':base_probs, 'Prob_Macro_Futures':futures_probs})
comparison_v5.to_csv('vix_futures_cboe/VIX_MODEL_COMPARISON_V5.csv', index=False)


## 8. V6 — structure à terme à maturité constante et validation walk-forward

Interpolation **sans extrapolation** entre les deux contrats encadrant 30 et 60 jours (jours calendaires avant échéance). Les valeurs manquantes restent manquantes. Variations sur 5 et 20 séances, puis décalage d'une séance pour éviter le lookahead. Comparaison macro / futures bruts / futures enrichis sur **les mêmes dates** et les mêmes fenêtres de test. Les fenêtres sont chronologiques et purgées de cinq séances SPX.


In [ ]:
# V6 : features construites sur les cotations VX et dates d'expiration originales.
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

vx_v6 = vx.copy()
def constant_maturity(row, target_days):
    pairs = []
    for i in range(1, 5):
        price, expiry = row.get(f'VX{i}'), row.get(f'VX{i}_Expiry')
        if pd.notna(price) and pd.notna(expiry) and price > 0:
            days = (expiry - row.name).days
            if days > 0:
                pairs.append((days, float(price)))
    pairs.sort()
    for (d0, p0), (d1, p1) in zip(pairs, pairs[1:]):
        if d0 <= target_days <= d1 and d1 > d0:
            return p0 + (p1 - p0) * (target_days - d0) / (d1 - d0)
    return np.nan

for days in (30, 60):
    vx_v6[f'VX_CM{days}'] = vx_v6.apply(constant_maturity, axis=1, target_days=days)
vx_v6['VX_CM_Slope_30_60'] = vx_v6['VX_CM60'] - vx_v6['VX_CM30']
vx_v6['VX_CM_Contango_30_60'] = vx_v6['VX_CM60'] / vx_v6['VX_CM30'] - 1
for feature in ('VX_Slope_1_2', 'VX_Contango_1_2', 'VX_CM_Slope_30_60', 'VX_CM_Contango_30_60'):
    for lag in (5, 20):
        # Évite les différences traversant des trous de données.
        vx_v6[f'{feature}_Delta_{lag}D'] = vx_v6[feature].diff(lag)

V6_EXTRA_RAW = ['VX_CM30', 'VX_CM60', 'VX_CM_Slope_30_60', 'VX_CM_Contango_30_60'] + [
    f'{name}_Delta_{lag}D'
    for name in ('VX_Slope_1_2', 'VX_Contango_1_2', 'VX_CM_Slope_30_60', 'VX_CM_Contango_30_60')
    for lag in (5, 20)
]
v6_features = vx_v6[V6_EXTRA_RAW].shift(1).add_prefix('FUT_').reindex(data.index)
v6_dataset = futures_dataset.join(v6_features, how='left')
# Même population pour les trois modèles ; l'interpolation peut réduire la couverture.
V6_EXTRA = list(v6_features.columns)
v6_dataset = v6_dataset.dropna(subset=HISTORICAL_FEATURES + EARLY_VX_FEATURES + V6_EXTRA + ['Crash_Target_Fwd5D'])
print('V6 maturités constantes :', len(v6_dataset), 'séances,',
      v6_dataset.index.min().date(), '→', v6_dataset.index.max().date())
if len(v6_dataset) < 1000:
    raise ValueError('V6 : couverture insuffisante des maturités constantes, inspecter VX1–VX4 et expirations.')

v6_sets = {
    'Macro': HISTORICAL_FEATURES,
    'Macro + VX1/VX2': HISTORICAL_FEATURES + EARLY_VX_FEATURES,
    'Macro + VX enrichis': HISTORICAL_FEATURES + EARLY_VX_FEATURES + V6_EXTRA,
}
# Fenêtres de test non chevauchantes, 3 ans d'entraînement minimum, puis 1 an par test.
# Les dates de test sont définies à l'avance, sans sélection selon les résultats.
first_test_year = max(2011, v6_dataset.index.min().year + 4)
folds = []
for year in range(first_test_year, v6_dataset.index.max().year + 1):
    test_idx = v6_dataset.index[(v6_dataset.index.year == year)]
    if len(test_idx) < 60:
        continue
    test_start = test_idx[0]
    spx_pos = data.index.get_indexer([test_start])[0]
    if spx_pos < CRASH_DAYS:
        continue
    safe_end = data.index[spx_pos - CRASH_DAYS]
    tr = v6_dataset.loc[v6_dataset.index < safe_end]
    te = v6_dataset.loc[test_idx]
    if len(tr) < 600 or tr['Crash_Target_Fwd5D'].nunique() != 2:
        continue
    if te['Crash_Target_Fwd5D'].nunique() != 2:
        print('Test ignoré (une seule classe) :', year)
        continue
    folds.append((year, tr, te))
if len(folds) < 3:
    raise ValueError(f'Seulement {len(folds)} fenêtres valides : historique insuffisant.')
v6_rows, v6_predictions = [], []
for year, tr, te in folds:
    ytr, yte = tr['Crash_Target_Fwd5D'].astype(int), te['Crash_Target_Fwd5D'].astype(int)
    for name, features in v6_sets.items():
        model = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=.035,
                              subsample=.8, colsample_bytree=.85,
                              eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(tr[features], ytr)
        prob = model.predict_proba(te[features])[:, 1]
        v6_rows.append({'Année_test': year, 'Modèle': name, 'N_test': len(te),
                        'N_crash': int(yte.sum()), 'AUC': roc_auc_score(yte, prob),
                        'AP': average_precision_score(yte, prob),
                        'Brier': brier_score_loss(yte, prob)})
        v6_predictions.extend({'Date': dt, 'Année_test': year, 'Modèle': name,
                               'Crash_Observed': int(y), 'Probabilité': float(p)}
                              for dt, y, p in zip(te.index, yte, prob))
v6_results = pd.DataFrame(v6_rows)
display(v6_results.pivot(index='Année_test', columns='Modèle', values=['AUC', 'AP', 'Brier']).round(4))
print('Moyennes par fenêtre (non pondérées) :')
display(v6_results.groupby('Modèle')[['AUC', 'AP', 'Brier']].mean().round(4))
outdir = Path('vix_futures_cboe')
outdir.mkdir(exist_ok=True)
v6_results.to_csv(outdir / 'VIX_MODEL_WALKFORWARD_V6.csv', index=False)
pd.DataFrame(v6_predictions).to_csv(outdir / 'VIX_MODEL_PREDICTIONS_V6.csv', index=False)
print('Exports V6 enregistrés dans', outdir)


## 9. V7 — baisse de 2 % à un moment quelconque des cinq séances suivantes

Nouvelle cible : le **minimum des cinq clôtures futures** est inférieur ou égal à 98 % de la clôture courante. La séance courante est exclue, et les cinq clôtures futures doivent être connues. Comparaison des mêmes trois ensembles de variables que V6, sur les **mêmes dates et fenêtres walk-forward** ; purge de cinq séances SPX avant chaque année de test. Les résultats V6 et V7 ne mesurent pas exactement le même événement : comparer aussi la fréquence des événements.


In [ ]:
# Cible V7 : min(SPX[t+1], ..., SPX[t+5]) / SPX[t] - 1 <= -2%.
future_closes = pd.concat(
    [data['SPX'].shift(-k).rename(f'SPX_t_plus_{k}') for k in range(1, CRASH_DAYS + 1)],
    axis=1
)
future_min = future_closes.min(axis=1, skipna=False)
v7_target = (future_min / data['SPX'] - 1).le(CRASH_FLAG).astype(float)
v7_target = v7_target.where(future_min.notna())
v7_dataset = v6_dataset.copy()
v7_dataset['Crash_Target_Any5D'] = v7_target.reindex(v7_dataset.index)
v7_dataset = v7_dataset.dropna(subset=['Crash_Target_Any5D'])
print('V7 échantillon :', len(v7_dataset), 'séances,',
      v7_dataset.index.min().date(), '→', v7_dataset.index.max().date())
print('Fréquence cible V6 (clôture J+5) :', f"{v7_dataset['Crash_Target_Fwd5D'].mean():.2%}")
print('Fréquence cible V7 (une des 5 clôtures) :', f"{v7_dataset['Crash_Target_Any5D'].mean():.2%}")

# Fenêtres annuelles identiques à V6, sauf éventuelles dernières dates sans cible connue.
v7_rows, v7_predictions = [], []
for year, _, _ in folds:
    te = v7_dataset.loc[v7_dataset.index.year == year]
    if len(te) < 60:
        continue
    first_test = te.index[0]
    pos = data.index.get_loc(first_test)
    safe_end = data.index[pos - CRASH_DAYS]
    tr = v7_dataset.loc[v7_dataset.index < safe_end]
    ytr = tr['Crash_Target_Any5D'].astype(int)
    yte = te['Crash_Target_Any5D'].astype(int)
    if len(tr) < 600 or ytr.nunique() != 2 or yte.nunique() != 2:
        print('V7 fenêtre ignorée :', year, '(classes ou historique insuffisants)')
        continue
    for name, features in v6_sets.items():
        model = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=.035,
                              subsample=.8, colsample_bytree=.85,
                              eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(tr[features], ytr)
        probs = model.predict_proba(te[features])[:, 1]
        v7_rows.append({'Année_test': year, 'Modèle': name, 'N_test': len(te),
                        'N_crash': int(yte.sum()), 'Taux_crash': float(yte.mean()),
                        'AUC': roc_auc_score(yte, probs),
                        'AP': average_precision_score(yte, probs),
                        'Brier': brier_score_loss(yte, probs)})
        v7_predictions.extend({'Date': dt, 'Année_test': year, 'Modèle': name,
                               'Crash_Observed': int(y), 'Probabilité': float(p)}
                              for dt, y, p in zip(te.index, yte, probs))
v7_results = pd.DataFrame(v7_rows)
if v7_results.empty:
    raise ValueError('Aucune fenêtre V7 valide.')
display(v7_results.pivot(index='Année_test', columns='Modèle', values=['AUC', 'AP', 'Brier']).round(4))
print('V7 — moyennes annuelles non pondérées :')
display(v7_results.groupby('Modèle')[['AUC', 'AP', 'Brier', 'Taux_crash']].mean().round(4))
print('V7 — comparaison avec V6 (événements différents) :')
display(pd.concat([
    v6_results.groupby('Modèle')[['AUC','AP','Brier']].mean().assign(Cible='V6 J+5'),
    v7_results.groupby('Modèle')[['AUC','AP','Brier']].mean().assign(Cible='V7 min J+1:J+5')
]).reset_index().set_index(['Cible','Modèle']).round(4))
v7_results.to_csv(outdir / 'VIX_MODEL_WALKFORWARD_V7.csv', index=False)
pd.DataFrame(v7_predictions).to_csv(outdir / 'VIX_MODEL_PREDICTIONS_V7.csv', index=False)
print('Exports V7 enregistrés dans', outdir)


## 10. V8 — précision des alertes, seuils et incertitude

Évaluation hors échantillon des probabilités V7 aux seuils fixes **20 %, 30 %, 40 %, 50 %** : précision, rappel, fréquence d'alerte, lift et matrice de confusion. Analyse du **top 10 %** des probabilités (seuil de rang calculé sur l'ensemble du test, descriptif seulement). Incertitude des écarts entre modèles par bootstrap apparié en blocs chronologiques de 20 séances (chevauchement des cibles sur cinq jours). Les seuils ne sont **pas optimisés sur le test**.


In [ ]:
# V8 — évaluer les prédictions walk-forward V7 sans réentraîner les modèles.
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss
pred = pd.DataFrame(v7_predictions).copy()
pred['Date'] = pd.to_datetime(pred['Date'])
pred = pred.sort_values(['Date', 'Modèle']).reset_index(drop=True)
thresholds = [0.20, 0.30, 0.40, 0.50]
rows = []
for name, grp in pred.groupby('Modèle'):
    y = grp['Crash_Observed'].to_numpy(dtype=int)
    p = grp['Probabilité'].to_numpy(dtype=float)
    prevalence = y.mean()
    for threshold in thresholds:
        alert = p >= threshold
        tp = int((alert & (y == 1)).sum())
        fp = int((alert & (y == 0)).sum())
        fn = int((~alert & (y == 1)).sum())
        tn = int((~alert & (y == 0)).sum())
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        rows.append({'Modèle': name, 'Seuil': threshold, 'N': len(y),
                     'Alertes': int(alert.sum()), 'Fréquence_alertes': alert.mean(),
                     'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                     'Précision': precision, 'Rappel': recall,
                     'Lift': precision / prevalence if prevalence and np.isfinite(precision) else np.nan})
v8_thresholds = pd.DataFrame(rows)
print('V8 — seuils de probabilité fixes, hors échantillon :')
display(v8_thresholds.round(4))

top_rows = []
for name, grp in pred.groupby('Modèle'):
    grp = grp.sort_values('Probabilité', ascending=False)
    n_top = max(1, int(np.ceil(len(grp) * .10)))
    selected = grp.head(n_top)
    base_rate = grp['Crash_Observed'].mean()
    top_rows.append({'Modèle': name, 'N_top10': n_top,
                     'Précision_top10': selected['Crash_Observed'].mean(),
                     'Taux_base': base_rate,
                     'Lift_top10': selected['Crash_Observed'].mean() / base_rate})
v8_top = pd.DataFrame(top_rows)
print('V8 — top 10 % des probabilités (descriptif, pas un seuil opérationnel) :')
display(v8_top.round(4))

# Bootstrap apparié par blocs de 20 séances : préserve les groupes d'événements qui se chevauchent.
wide = pred.pivot(index='Date', columns='Modèle', values='Probabilité').sort_index()
observed = pred.drop_duplicates('Date').set_index('Date')['Crash_Observed'].sort_index()
wide = wide.join(observed.rename('y'), how='inner').dropna()
model_names = [name for name in v6_sets if name in wide.columns]
y = wide['y'].to_numpy(dtype=int)
P = {name: wide[name].to_numpy(dtype=float) for name in model_names}
if len(model_names) != 3:
    raise ValueError('V8 : trois modèles attendus dans les prédictions V7.')
def metric_bundle(yy, pp):
    return np.array([roc_auc_score(yy, pp), average_precision_score(yy, pp),
                     brier_score_loss(yy, pp)])
baseline = 'Macro'
pairs = [('Macro + VX1/VX2', baseline), ('Macro + VX enrichis', baseline),
         ('Macro + VX enrichis', 'Macro + VX1/VX2')]
rng = np.random.default_rng(RANDOM_STATE)
block_size, iterations = 20, 1000
n = len(wide)
bootstrap = {pair: [] for pair in pairs}
for _ in range(iterations):
    # Blocs circulaires, même tirage de dates pour tous les modèles.
    starts = rng.integers(0, n, size=int(np.ceil(n / block_size)))
    indices = np.concatenate([(start + np.arange(block_size)) % n for start in starts])[:n]
    yy = y[indices]
    if np.unique(yy).size < 2:
        continue
    scores = {name: metric_bundle(yy, p[indices]) for name, p in P.items()}
    for pair in pairs:
        bootstrap[pair].append(scores[pair[0]] - scores[pair[1]])
ci_rows = []
for a, b in pairs:
    point = metric_bundle(y, P[a]) - metric_bundle(y, P[b])
    draws = np.asarray(bootstrap[(a, b)])
    for j, metric in enumerate(('AUC', 'AP', 'Brier')):
        lo, hi = np.percentile(draws[:, j], [2.5, 97.5])
        ci_rows.append({'Modèle_A': a, 'Modèle_B': b, 'Métrique': metric,
                        'Écart_A_moins_B': point[j], 'IC95_bas': lo, 'IC95_haut': hi,
                        'IC_exclut_zéro': bool(lo > 0 or hi < 0)})
v8_ci = pd.DataFrame(ci_rows)
print('V8 — écarts appariés et IC 95 % bootstrap en blocs (20 séances) :')
display(v8_ci.round(4))
print('Attention : les IC sont descriptifs, sans correction pour sélection multiple de modèles.')
v8_thresholds.to_csv(outdir / 'VIX_MODEL_THRESHOLDS_V8.csv', index=False)
v8_top.to_csv(outdir / 'VIX_MODEL_TOP10_V8.csv', index=False)
v8_ci.to_csv(outdir / 'VIX_MODEL_BOOTSTRAP_V8.csv', index=False)
print('Exports V8 enregistrés dans', outdir)


## 11. V9 — sélection automatique des variables : régression logistique L1 et Elastic Net

Même cible V7 et mêmes fenêtres annuelles hors échantillon. Standardisation et imputation **à l'intérieur** de chaque entraînement ; réglage de la régularisation par validation chronologique interne avec purge de cinq séances, sans utiliser les années de test. Les coefficients nuls sélectionnent les variables. Comparaison à XGBoost enrichi V7 sur les mêmes dates. Les résultats ne sont pas une garantie de performance future.


In [ ]:
# V9 — modèles parcimonieux vs XGBoost V7, sans fuite temporelle.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

V9_FEATURES = list(dict.fromkeys(HISTORICAL_FEATURES + EARLY_VX_FEATURES + V6_EXTRA))
V9_C_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
V9_PENALTIES = {'Logistique L1': 1.0, 'Logistique Elastic Net': 0.5}

def make_v9_pipeline(C, l1_ratio):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            penalty='elasticnet', l1_ratio=l1_ratio, C=C,
            solver='saga', max_iter=6000, tol=1e-3,
            random_state=RANDOM_STATE))
    ])

def select_v9_C(X, y, l1_ratio):
    # Le gap est mesuré en observations du dataset filtré (au moins cinq séances SPX).
    splits = list(TimeSeriesSplit(n_splits=3, gap=CRASH_DAYS).split(X))
    best_C, best_score = None, -np.inf
    for C in V9_C_GRID:
        scores = []
        for train_i, valid_i in splits:
            yt, yv = y.iloc[train_i], y.iloc[valid_i]
            if yt.nunique() < 2 or yv.nunique() < 2:
                continue
            pipeline = make_v9_pipeline(C, l1_ratio)
            pipeline.fit(X.iloc[train_i], yt)
            probs = pipeline.predict_proba(X.iloc[valid_i])[:, 1]
            scores.append(average_precision_score(yv, probs))
        if scores and np.mean(scores) > best_score:
            best_C, best_score = C, float(np.mean(scores))
    if best_C is None:
        raise ValueError('V9 : validation interne impossible (classes insuffisantes).')
    return best_C, best_score

v9_rows, v9_predictions, v9_coefficients = [], [], []
for year, _, _ in folds:
    test = v7_dataset.loc[v7_dataset.index.year == year]
    if len(test) < 60:
        continue
    test_start = test.index[0]
    safe_train_end = data.index[data.index.get_loc(test_start) - CRASH_DAYS]
    train = v7_dataset.loc[v7_dataset.index < safe_train_end]
    if len(train) < 600:
        continue
    y_train = train['Crash_Target_Any5D'].astype(int)
    y_test = test['Crash_Target_Any5D'].astype(int)
    if y_train.nunique() != 2 or y_test.nunique() != 2:
        continue
    for name, l1_ratio in V9_PENALTIES.items():
        best_C, cv_ap = select_v9_C(train[V9_FEATURES], y_train, l1_ratio)
        model = make_v9_pipeline(best_C, l1_ratio)
        model.fit(train[V9_FEATURES], y_train)
        probabilities = model.predict_proba(test[V9_FEATURES])[:, 1]
        coefficients = model.named_steps['classifier'].coef_[0]
        selected = int(np.count_nonzero(np.abs(coefficients) > 1e-8))
        for feature, coefficient in zip(V9_FEATURES, coefficients):
            v9_coefficients.append({'Année_test': year, 'Modèle': name,
                                    'Variable': feature, 'Coefficient_standardisé': float(coefficient),
                                    'Sélectionnée': bool(abs(coefficient) > 1e-8),
                                    'C_optimal': best_C})
        v9_rows.append({'Année_test': year, 'Modèle': name, 'N_test': len(test),
                        'N_crash': int(y_test.sum()), 'C_optimal': best_C,
                        'AP_validation_interne': cv_ap, 'N_variables_retenues': selected,
                        'AUC': roc_auc_score(y_test, probabilities),
                        'AP': average_precision_score(y_test, probabilities),
                        'Brier': brier_score_loss(y_test, probabilities)})
        v9_predictions.extend({'Date': date, 'Année_test': year, 'Modèle': name,
                               'Crash_Observed': int(observed), 'Probabilité': float(prob)}
                              for date, observed, prob in zip(test.index, y_test, probabilities))
v9_results = pd.DataFrame(v9_rows)
v9_pred = pd.DataFrame(v9_predictions)
v9_coefs = pd.DataFrame(v9_coefficients)
print('V9 — validation walk-forward annuelle :')
display(v9_results.pivot(index='Année_test', columns='Modèle', values=['AUC','AP','Brier','N_variables_retenues']).round(4))
print('V9 — moyennes annuelles non pondérées :')
display(v9_results.groupby('Modèle')[['AUC','AP','Brier','N_variables_retenues']].mean().round(4))
print('V9 — fréquence de sélection des variables (par année et modèle) :')
display(v9_coefs.groupby(['Modèle','Variable'])['Sélectionnée'].mean()
        .unstack(0).sort_values('Logistique L1', ascending=False).round(2))
# Comparaison appariée avec les prédictions XGBoost V7, sur dates communes.
xgb_v7 = pd.DataFrame(v7_predictions)
xgb_v7 = xgb_v7.loc[xgb_v7['Modèle'] == 'Macro + VX enrichis',
                    ['Date','Crash_Observed','Probabilité']].copy()
xgb_v7['Date'] = pd.to_datetime(xgb_v7['Date'])
xgb_v7['Modèle'] = 'XGBoost V7 enrichi'
v9_compare = pd.concat([v9_pred, xgb_v7], ignore_index=True)
v9_compare['Date'] = pd.to_datetime(v9_compare['Date'])
common_dates = set.intersection(*(set(g['Date']) for _, g in v9_compare.groupby('Modèle')))
v9_compare = v9_compare.loc[v9_compare['Date'].isin(common_dates)].copy()
summary = []
for name, g in v9_compare.groupby('Modèle'):
    y = g['Crash_Observed'].to_numpy(dtype=int)
    p = g['Probabilité'].to_numpy(dtype=float)
    top_n = max(1, int(np.ceil(len(g)*0.10)))
    top_precision = g.nlargest(top_n, 'Probabilité')['Crash_Observed'].mean()
    summary.append({'Modèle': name, 'N': len(g), 'AUC': roc_auc_score(y,p),
                    'AP': average_precision_score(y,p), 'Brier': brier_score_loss(y,p),
                    'Précision_top10': top_precision, 'Taux_crash': y.mean()})
v9_summary = pd.DataFrame(summary).sort_values('AP', ascending=False)
print('V9 — comparaison globale à dates identiques :')
display(v9_summary.round(4))
outdir = Path('vix_futures_cboe')
outdir.mkdir(exist_ok=True)
v9_results.to_csv(outdir / 'VIX_MODEL_WALKFORWARD_V9.csv', index=False)
v9_pred.to_csv(outdir / 'VIX_MODEL_PREDICTIONS_V9.csv', index=False)
v9_coefs.to_csv(outdir / 'VIX_MODEL_COEFFICIENTS_V9.csv', index=False)
v9_summary.to_csv(outdir / 'VIX_MODEL_COMPARISON_V9.csv', index=False)
print('Exports V9 enregistrés dans', outdir)


## 12. V10 — probabilité d'un krach de plus de 5 % dans les N prochaines séances

Un krach est une **baisse strictement supérieure à 5 % entre deux clôtures chronologiquement ordonnées quelconques** de la fenêtre `T, T+1, ..., T+N`. La date exacte n'est pas prédite. Horizons : **5, 10, 15, 20 séances**. Modèle : logistique L1, régularisation choisie uniquement sur le passé (validation interne chronologique). Purge de **N séances SPX** avant chaque test annuel. Les taux de base diffèrent selon l'horizon : comparer l'AUC, le lift et le rappel, pas seulement la précision brute.

**Anticipation** : pour les événements positifs, on distingue ceux dont le sommet du segment de chute survient **après T** de ceux où le sommet est déjà à T ; ce diagnostic n'est pas une garantie de signal avant la baisse. Les métriques d'alerte top 10 % sont descriptives (seuil calculé sur les prédictions hors échantillon).


In [ ]:
# V10 : maximum drawdown futur sur T..T+N, sans imposer le sommet à T.
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
# USHY (FRED BAMLH0A0HYM2) est déjà un spread OAS, pas un rendement obligataire.
# Écarter HY_minus_UST3Y : soustraire le Treasury à un OAS est incorrect.
V10_FEATURES = [f for f in V9_FEATURES if f != 'HY_minus_UST3Y']
V10_FEATURES = ['USHY_OAS' if f == 'USHY_Yield' else
                'USHY_OAS_Change_5D' if f == 'USHY_Change_5D' else f
                for f in V10_FEATURES]
V10_HORIZONS = [5, 10, 15, 20]
V10_DROP = -0.05  # strictement inférieur à -5 %
v10_close = data['SPX'].to_numpy(dtype=float)
v10_index = data.index
v10_targets = {}
v10_peak_lags = {}
v10_trough_lags = {}
for horizon in V10_HORIZONS:
    labels = np.full(len(v10_close), np.nan)
    peak_lags = np.full(len(v10_close), np.nan)
    trough_lags = np.full(len(v10_close), np.nan)
    for t in range(len(v10_close) - horizon):
        window = v10_close[t:t + horizon + 1]
        if not np.isfinite(window).all() or np.any(window <= 0):
            continue
        running_peak = window[0]
        running_peak_i = 0
        min_drawdown = 0.0
        best_peak_i, best_trough_i = 0, 0
        for j in range(1, len(window)):
            decline = window[j] / running_peak - 1
            if decline < min_drawdown:
                min_drawdown = decline
                best_peak_i, best_trough_i = running_peak_i, j
            if window[j] > running_peak:
                running_peak = window[j]
                running_peak_i = j
        labels[t] = float(min_drawdown < V10_DROP)
        if labels[t] == 1:
            peak_lags[t] = best_peak_i
            trough_lags[t] = best_trough_i
    v10_targets[horizon] = pd.Series(labels, index=v10_index)
    v10_peak_lags[horizon] = pd.Series(peak_lags, index=v10_index)
    v10_trough_lags[horizon] = pd.Series(trough_lags, index=v10_index)

v10_rows, v10_preds, v10_coef = [], [], []
for horizon in V10_HORIZONS:
    ds = v6_dataset.copy()
    ds = ds.rename(columns={'USHY_Yield': 'USHY_OAS', 'USHY_Change_5D': 'USHY_OAS_Change_5D'})
    ds['Crash_MDD'] = v10_targets[horizon].reindex(ds.index)
    ds = ds.dropna(subset=['Crash_MDD'])
    print(f'Horizon {horizon}j : {len(ds)} séances, taux de crash {ds.Crash_MDD.mean():.2%}')
    for year in range(max(2011, ds.index.min().year + 4), ds.index.max().year + 1):
        test = ds.loc[ds.index.year == year]
        if len(test) < 60:
            continue
        start_pos = data.index.get_loc(test.index[0])
        if start_pos < horizon:
            continue
        # Dernière date d'entraînement : cible entièrement connue AVANT le début du test.
        cutoff = data.index[start_pos - horizon]
        train = ds.loc[ds.index < cutoff]
        if len(train) < 600:
            continue
        y_train = train['Crash_MDD'].astype(int)
        y_test = test['Crash_MDD'].astype(int)
        if y_train.nunique() < 2 or y_test.nunique() < 2:
            print('V10 fenêtre ignorée (une seule classe) :', horizon, year)
            continue
        # Sélection de C uniquement dans l'entraînement, avec purge de N observations.
        X_train = train[V10_FEATURES]
        inner_splits = list(TimeSeriesSplit(n_splits=3, gap=horizon).split(X_train))
        best_C, best_ap = None, -np.inf
        for C in V9_C_GRID:
            cv_scores = []
            for it, iv in inner_splits:
                if y_train.iloc[it].nunique() < 2 or y_train.iloc[iv].nunique() < 2:
                    continue
                candidate = make_v9_pipeline(C, 1.0)
                candidate.fit(X_train.iloc[it], y_train.iloc[it])
                cv_scores.append(average_precision_score(
                    y_train.iloc[iv], candidate.predict_proba(X_train.iloc[iv])[:, 1]))
            if cv_scores and np.mean(cv_scores) > best_ap:
                best_C, best_ap = C, float(np.mean(cv_scores))
        if best_C is None:
            continue
        model = make_v9_pipeline(best_C, 1.0)
        model.fit(X_train, y_train)
        probabilities = model.predict_proba(test[V10_FEATURES])[:, 1]
        coefficients = model.named_steps['classifier'].coef_[0]
        v10_rows.append({'Horizon': horizon, 'Année_test': year, 'N_test': len(test),
                         'N_crash': int(y_test.sum()), 'Taux_crash': y_test.mean(),
                         'AUC': roc_auc_score(y_test, probabilities),
                         'AP': average_precision_score(y_test, probabilities),
                         'Brier': brier_score_loss(y_test, probabilities),
                         'C_optimal': best_C,
                         'N_variables': int((np.abs(coefficients) > 1e-8).sum())})
        for feature, coefficient in zip(V10_FEATURES, coefficients):
            v10_coef.append({'Horizon': horizon, 'Année_test': year,
                             'Variable': feature, 'Coefficient': float(coefficient),
                             'Sélectionnée': bool(abs(coefficient) > 1e-8)})
        for date, observed, probability in zip(test.index, y_test, probabilities):
            v10_preds.append({'Date': date, 'Horizon': horizon,
                              'Année_test': year, 'Crash_Observed': int(observed),
                              'Probabilité': float(probability),
                              'Sommet_après_T': bool(v10_peak_lags[horizon].loc[date] > 0)
                              if observed else np.nan,
                              'Délai_vers_sommet': v10_peak_lags[horizon].loc[date],
                              'Délai_vers_creux': v10_trough_lags[horizon].loc[date]})
v10_results = pd.DataFrame(v10_rows)
v10_predictions = pd.DataFrame(v10_preds)
v10_coefficients = pd.DataFrame(v10_coef)
if v10_results.empty:
    raise ValueError('V10 : aucune fenêtre de validation exploitable.')
print('V10 — moyennes annuelles (non pondérées) :')
display(v10_results.groupby('Horizon')[['AUC','AP','Brier','Taux_crash','N_variables']].mean().round(4))
v10_summary = []
for horizon, g in v10_predictions.groupby('Horizon'):
    y = g['Crash_Observed'].to_numpy(dtype=int)
    p = g['Probabilité'].to_numpy(dtype=float)
    top_n = max(1, int(np.ceil(.10 * len(g))))
    selected = g.nlargest(top_n, 'Probabilité')
    precision = selected['Crash_Observed'].mean()
    recall = selected['Crash_Observed'].sum() / y.sum() if y.sum() else np.nan
    pre_peak = selected.loc[selected['Crash_Observed'] == 1, 'Sommet_après_T']
    v10_summary.append({'Horizon': horizon, 'N_test': len(g), 'N_crash': int(y.sum()),
                        'Taux_crash': y.mean(), 'AUC': roc_auc_score(y, p),
                        'AP': average_precision_score(y, p),
                        'Brier': brier_score_loss(y, p),
                        'Précision_top10': precision, 'Rappel_top10': recall,
                        'Lift_top10': precision / y.mean(),
                        'Part_alertes_positives_avant_sommet': pre_peak.mean() if len(pre_peak) else np.nan})
v10_summary = pd.DataFrame(v10_summary).sort_values('Horizon')
print('V10 — comparaison consolidée des horizons (hors échantillon) :')
display(v10_summary.round(4))
print('V10 — variables sélectionnées, fréquence par horizon :')
display(v10_coefficients.groupby(['Variable','Horizon'])['Sélectionnée'].mean()
        .unstack().fillna(0).round(2).sort_values(10, ascending=False))
outdir = Path('vix_futures_cboe')
outdir.mkdir(exist_ok=True)
v10_results.to_csv(outdir / 'VIX_MODEL_WALKFORWARD_V10.csv', index=False)
v10_predictions.to_csv(outdir / 'VIX_MODEL_PREDICTIONS_V10.csv', index=False)
v10_coefficients.to_csv(outdir / 'VIX_MODEL_COEFFICIENTS_V10.csv', index=False)
v10_summary.to_csv(outdir / 'VIX_MODEL_COMPARISON_V10.csv', index=False)
print('Exports V10 enregistrés dans', outdir)


## 13. V11 — impact de l'épisode Covid sur l'entraînement (5 et 10 séances)

**Expérience contrôlée** : pour chaque année test 2021–2026, comparer trois entraînements, sans changer les observations de test ni les variables : A (complet), B (sans 2020-02-01 à 2020-06-30), C (sans toute l'année 2020). La sélection de la pénalisation L1 est effectuée séparément sur chaque historique passé. La purge temporelle précède toute exclusion de dates ; le découpage est fait sur le calendrier SPX original. Une année sans événements reste évaluée dans les prédictions consolidées (AUC annuelle non définie). Les périodes 2011–2020 sont hors du test causal de l'exclusion, car les données Covid n'étaient pas encore disponibles.

**Interprétation** : comparer surtout AUC, AP, Brier, précision et lift top 10 % hors échantillon. Les labels se chevauchent : ne pas traiter chaque journée positive comme une crise indépendante. La comparaison par épisodes distincts nécessitera un diagnostic complémentaire.


In [ ]:
# V11 — entraînement complet / sans épisode Covid / sans 2020.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

V11_HORIZONS = [5, 10]
V11_SCENARIOS = {
    'A_Complet': None,
    'B_Sans_Covid_Fev_Juin': ('2020-02-01', '2020-06-30'),
    'C_Sans_2020': ('2020-01-01', '2020-12-31'),
}
V11_TEST_YEARS = range(2021, 2027)
V11_FEATURES = list(V10_FEATURES)
V11_C_GRID = list(V9_C_GRID)

def v11_model(C):
    # Liblinear est rapide et résout directement la pénalisation L1.
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            penalty='l1', C=C, solver='liblinear',
            max_iter=2500, random_state=RANDOM_STATE))
    ])

v11_folds, v11_predictions, v11_coefficients = [], [], []
for horizon in V11_HORIZONS:
    ds = v6_dataset.copy().rename(columns={
        'USHY_Yield': 'USHY_OAS', 'USHY_Change_5D': 'USHY_OAS_Change_5D'})
    ds['Crash_MDD'] = v10_targets[horizon].reindex(ds.index)
    ds = ds.dropna(subset=['Crash_MDD']).sort_index()
    for year in V11_TEST_YEARS:
        test = ds.loc[ds.index.year == year]
        if len(test) < 60:
            continue
        start_pos = data.index.get_loc(test.index[0])
        # Purge de N séances sur le calendrier SPX AVANT de retirer les dates Covid.
        cutoff = data.index[start_pos - horizon]
        eligible = ds.loc[ds.index < cutoff]
        y_test = test['Crash_MDD'].astype(int)
        for scenario, excluded in V11_SCENARIOS.items():
            train = eligible
            if excluded is not None:
                train = train.loc[~((train.index >= pd.Timestamp(excluded[0])) &
                                    (train.index <= pd.Timestamp(excluded[1])))]
            if len(train) < 600 or train['Crash_MDD'].nunique() < 2:
                print('V11 entraînement insuffisant :', horizon, year, scenario)
                continue
            X_train, y_train = train[V11_FEATURES], train['Crash_MDD'].astype(int)
            # Validation interne chronologique, gap N sur le jeu d'entraînement filtré.
            splits = list(TimeSeriesSplit(n_splits=3, gap=horizon).split(X_train))
            best_C, best_ap = None, -np.inf
            for C in V11_C_GRID:
                scores = []
                for it, iv in splits:
                    if y_train.iloc[it].nunique() < 2 or y_train.iloc[iv].nunique() < 2:
                        continue
                    m = v11_model(C)
                    m.fit(X_train.iloc[it], y_train.iloc[it])
                    scores.append(average_precision_score(
                        y_train.iloc[iv], m.predict_proba(X_train.iloc[iv])[:, 1]))
                if scores and float(np.mean(scores)) > best_ap:
                    best_C, best_ap = C, float(np.mean(scores))
            if best_C is None:
                print('V11 pas de validation interne :', horizon, year, scenario)
                continue
            model = v11_model(best_C)
            model.fit(X_train, y_train)
            probs = model.predict_proba(test[V11_FEATURES])[:, 1]
            auc = roc_auc_score(y_test, probs) if y_test.nunique() > 1 else np.nan
            v11_folds.append({
                'Horizon': horizon, 'Scenario': scenario, 'Annee_test': year,
                'N_train': len(train), 'N_test': len(test), 'N_crash': int(y_test.sum()),
                'Taux_crash': y_test.mean(), 'AUC': auc,
                'AP': average_precision_score(y_test, probs),
                'Brier': brier_score_loss(y_test, probs), 'C_optimal': best_C})
            for date, observed, prob in zip(test.index, y_test, probs):
                v11_predictions.append({
                    'Date': date, 'Horizon': horizon, 'Scenario': scenario,
                    'Annee_test': year, 'Crash_Observed': int(observed),
                    'Probabilite': float(prob)})
            for feature, coef in zip(V11_FEATURES, model.named_steps['classifier'].coef_[0]):
                v11_coefficients.append({
                    'Horizon': horizon, 'Scenario': scenario, 'Annee_test': year,
                    'Variable': feature, 'Coefficient': float(coef),
                    'Selectionnee': int(abs(coef) > 1e-8)})
        print('V11 terminé : horizon', horizon, 'année', year)

v11_folds = pd.DataFrame(v11_folds)
v11_predictions = pd.DataFrame(v11_predictions)
v11_coefficients = pd.DataFrame(v11_coefficients)
if v11_predictions.empty:
    raise ValueError('V11 : aucune prédiction produite.')
# Comparaison strictement appariée : ne conserver que les dates prédites par les 3 scénarios.
counts = v11_predictions.groupby(['Horizon', 'Date'])['Scenario'].nunique()
common_keys = counts[counts == len(V11_SCENARIOS)].index
v11_predictions = v11_predictions.set_index(['Horizon', 'Date']).loc[common_keys].reset_index()
v11_comparison = []
for (horizon, scenario), g in v11_predictions.groupby(['Horizon', 'Scenario']):
    g = g.sort_values('Date')
    y, p = g['Crash_Observed'].to_numpy(int), g['Probabilite'].to_numpy(float)
    top_n = max(1, int(np.ceil(len(g) * 0.10)))
    top = g.nlargest(top_n, 'Probabilite')
    precision = top['Crash_Observed'].mean()
    base = y.mean()
    v11_comparison.append({
        'Horizon': horizon, 'Scenario': scenario, 'N_test': len(g),
        'N_crash': int(y.sum()), 'Taux_crash': base,
        'AUC': roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        'AP': average_precision_score(y, p),
        'Brier': brier_score_loss(y, p),
        'Precision_top10': precision,
        'Rappel_top10': top['Crash_Observed'].sum() / y.sum() if y.sum() else np.nan,
        'Lift_top10': precision / base if base else np.nan})
v11_comparison = pd.DataFrame(v11_comparison).sort_values(['Horizon', 'Scenario'])
print('V11 — comparaison appariée, tests 2021–2026 :')
display(v11_comparison.round(4))
print('V11 — performance annuelle :')
display(v11_folds.pivot_table(index=['Horizon','Annee_test'],
                             columns='Scenario', values='AUC').round(3))
print('V11 — sélection des variables High Yield :')
display(v11_coefficients.loc[v11_coefficients.Variable.str.contains('USHY')]
        .groupby(['Horizon','Scenario','Variable'])['Selectionnee'].mean().round(3))
outdir = Path('vix_futures_cboe')
outdir.mkdir(exist_ok=True)
v11_comparison.to_csv(outdir / 'VIX_MODEL_COMPARISON_V11.csv', index=False)
v11_folds.to_csv(outdir / 'VIX_MODEL_WALKFORWARD_V11.csv', index=False)
v11_predictions.to_csv(outdir / 'VIX_MODEL_PREDICTIONS_V11.csv', index=False)
v11_coefficients.to_csv(outdir / 'VIX_MODEL_COEFFICIENTS_V11.csv', index=False)
print('Exports V11 disponibles dans', outdir)
